# v03 — Routed Few-shot

End-to-end Colab runner for `v03_routed_fewshot`. Two-pass pipeline:
1. **Pass 1**: Classify each question into (domain, answer_type) using the model
2. **Pass 2**: Solve with domain-specific system prompt + matched few-shot examples

All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

## 1. Mount Drive + chdir

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

Mounted at /content/drive
cwd: /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils


In [2]:
!pip -q install -r app/physics_solution/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 142.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.9.0 which is incompatible.


### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [3]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

[install_fast_kernels] env: python=cp312 torch=2.8 cuda=12.6 -> tag cu12
[install_fast_kernels] flash-linear-attention ...
  fla: OK
[install_fast_kernels] causal-conv1d (pre-built wheels) ...
  trying v1.5.2 abi=FALSE ...
  causal-conv1d: OK (1.5.2, abi=FALSE)


{'fla': True, 'causal_conv1d': True}

In [4]:
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

[colab_setup] loaded /content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils/app/physics_solution/.env
               cwd: /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils
          hf_token: OK
         langsmith: OFF
        base_model: Qwen/Qwen3.5-4B
             dtype: bf16
            device: cuda
        batch_size: 100
    max_new_tokens: 640
       temperature: 0.0
         test_file: app/physics_solution/data/test/full_test.csv


{'cwd': '/content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils',
 'hf_token': 'OK',
 'langsmith': 'OFF',
 'base_model': 'Qwen/Qwen3.5-4B',
 'dtype': 'bf16',
 'device': 'cuda',
 'batch_size': 100,
 'max_new_tokens': 640,
 'temperature': 0.0,
 'test_file': 'app/physics_solution/data/test/full_test.csv'}

In [3]:
!nvidia-smi -L

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-0bedfb69-04c2-449f-77ee-a7fa758fc1cd)


## 2. Test set paths

Two evaluation runs: **full test** (original data) and **golden** (DeepSeek-rewritten CoT).

| File | Rows | Scope |
|---|---|---|
| `full_test.csv` | 1352 | All answer types — original CoT |
| `deepseek-v4-pro_golden_data.csv` | 1352 | All answer types — golden CoT |

In [5]:
import os

# --- Full test (original data) ---
FULL_TEST_FILE  = 'app/physics_solution/data/test/full_test.csv'
FULL_OUT_FILE   = 'app/physics_solution/versions/v03_routed_fewshot/output/results_full.json'

# --- Golden test (DeepSeek-rewritten CoT) ---
GOLDEN_TEST_FILE = 'app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv'
GOLDEN_OUT_FILE  = 'app/physics_solution/versions/v03_routed_fewshot/output/results_golden.json'

for label, f in [('Full', FULL_TEST_FILE), ('Golden', GOLDEN_TEST_FILE)]:
    assert os.path.exists(f), f"Not found: {f}"
    print(f'{label}: {f}')

Full: app/physics_solution/data/test/full_test.csv
Golden: app/physics_solution/data/golden/deepseek-v4-pro_golden_data.csv


## 3. Curate few-shot examples

Picks K examples per (domain, answer_type) group. Excludes `full_test.csv` IDs to prevent leakage.

In [6]:
!python -m app.physics_solution.versions.v03_routed_fewshot.select_fewshot --k 3 --seed 42

Excluding 0 test IDs
Candidate pool: 1352 rows across 8 domains, 6 answer types
Groups:
__prefix  __answer_type
CH        mixed              1
          pure_numeric     288
          sci_notation       1
CHLT      yes_no            20
DDT       mixed              4
          multi_value        1
          pure_numeric      78
          sci_notation      21
          text_only         25
          yes_no             1
DT        mixed              6
          pure_numeric      39
          sci_notation      23
LD        mixed              5
          pure_numeric     201
          sci_notation     190
          text_only          1
NL        mixed              8
          pure_numeric     160
          text_only         22
TD        mixed              1
          multi_value        2
          pure_numeric     166
          sci_notation       4
          text_only          4
THCB      multi_value       23
          pure_numeric      53
          text_only          4

Wrote 71 examples t

## 4. Inference (2-pass: classify → route → solve)

Runs both full test and golden test sequentially. All knobs (batch size, dtype, max_new_tokens, ...) come from `config.py`.

In [11]:
# --- 4a. Full test ---
!python -m app.physics_solution.cli.inference \
    --version v03_routed_fewshot \
    --test-file {FULL_TEST_FILE} \
    --out {FULL_OUT_FILE} \
    --n-examples 3 \
    --batch-size 80

[v03_routed_fewshot] loading Qwen/Qwen3.5-4B dtype=bf16 device=cuda batch_size=80
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100% 2/2 [00:00<00:00, 25731.93it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 426/426 [00:00<00:00, 6627.35it/s]
[v03_routed_fewshot] loaded 71 few-shot examples
[v03_routed_fewshot] running on 1352 questions

--- Pass 1: classifying 1352 questions ---
Classification done in 52.9s
Domain distribution: {'DT': 218, 'LD': 248, 'CH': 290, 'NL': 159, 'CHLT': 5, 'TD': 252, 'DDT': 120, 'THCB': 60}
Answer type distribution: {'pure_numeric': 1118, 'mixed': 56, 'text_only': 113, 'sci_notation': 16, 'multi_value': 26, 'yes_no': 23}

--- Pass 2: solving 1352 questions ---
  0% 0/1352 [00:00<?, ?it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
  6% 80/1352 [01:17<20:25,  1.04it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 12% 160/1352 [02:17<16:43,  1.19it/s]  [HF

In [12]:
# --- 4b. Golden test ---
!python -m app.physics_solution.cli.inference \
    --version v03_routed_fewshot \
    --test-file {GOLDEN_TEST_FILE} \
    --out {GOLDEN_OUT_FILE} \
    --n-examples 3 \
    --batch-size 80

[v03_routed_fewshot] loading Qwen/Qwen3.5-4B dtype=bf16 device=cuda batch_size=80
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100% 2/2 [00:00<00:00, 36954.22it/s]
Download complete: : 0.00B [00:00, ?B/s]
Loading weights: 100% 426/426 [00:00<00:00, 6365.72it/s]
[v03_routed_fewshot] loaded 71 few-shot examples
[v03_routed_fewshot] running on 1352 questions

--- Pass 1: classifying 1352 questions ---
Classification done in 53.0s
Domain distribution: {'DT': 218, 'LD': 248, 'CH': 290, 'NL': 159, 'CHLT': 5, 'TD': 252, 'DDT': 120, 'THCB': 60}
Answer type distribution: {'pure_numeric': 1118, 'mixed': 56, 'text_only': 113, 'sci_notation': 16, 'multi_value': 26, 'yes_no': 23}

--- Pass 2: solving 1352 questions ---
  0% 0/1352 [00:00<?, ?it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
  6% 80/1352 [01:03<16:54,  1.25it/s]  [HFBatchedLLM] generating batch of 80 prompts (max_new_tokens=640)
 12% 160/1352 [01:51<13:26,  1.48it/s]  [HF

## 5. Push to HF (with full metadata)

In [13]:
!python -m app.physics_solution.cli.upload_model \
    --version-num 3 --strategy routed_fewshot \
    --results {FULL_OUT_FILE} \
    --test-file {FULL_TEST_FILE} \
    --extra-results {GOLDEN_OUT_FILE}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Fetching 14 files:   0% 0/14 [00:00<?, ?it/s]
Fetching 14 files: 100% 14/14 [00:01<00:00,  9.27it/s]
Download complete: 100% 77.7k/77.7k [00:01<00:00, 55.1kB/s]                Snapshotted Qwen/Qwen3.5-4B -> /content/drive/.shortcut-targets-by-id/1QTSWNOYGynQ_AuaWq1qQpOp3mbD-KF75/NCKH/Exact_2026_Laplace-s_Red_Devils/.hf_snapshots/Qwen__Qwen3.5-4B
Download complete: 100% 77.7k/77.7k [00:02<00:00, 38.5kB/s]
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...Qwen3.5-4B/tokenizer.json: 100% 12.8M/12.8M [00:00<?, ?B/s]

Processing Files (1 / 3)      :   0% 12.8M/9.33G [00:00<07:42, 20.2MB/s,   ???B/s  ]


  ...0002-of-00002.safetenso

## 6. Inspect results

In [14]:
import json, pandas as pd

for label, path in [('FULL', FULL_OUT_FILE), ('GOLDEN', GOLDEN_OUT_FILE)]:
    print(f'\n{"="*60}')
    print(f'  {label}: {path}')
    print(f'{"="*60}')
    data = json.loads(open(path, encoding='utf-8').read())
    print('summary:', json.dumps(data['summary'], indent=2, ensure_ascii=False))
    df = pd.DataFrame(data['rows'])
    wrong = df[~df['is_correct']].copy()
    wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
    print(f"\nWrong: {len(wrong)} / {len(df)}")
    print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")
    display(wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'reached_final']].head(20))


  FULL: app/physics_solution/versions/v03_routed_fewshot/output/results_full.json
summary: {
  "n": 1352,
  "correct": 732,
  "accuracy": 0.5414201183431953,
  "mean_latency_s": 0.6252372414755398,
  "version": "v03_routed_fewshot",
  "model_id": "Qwen/Qwen3.5-4B",
  "dtype": "bf16",
  "effective_dtype": "bf16",
  "batch_size": 80,
  "assistant_prefix": "",
  "description": "Two-pass pipeline on the same model: Pass 1 classifies (domain, answer_type) from question text alone; Pass 2 solves with a domain-specific system prompt and answer-type-matched few-shot examples.",
  "classify_elapsed_s": 52.88875079154968,
  "n_examples": 3
}

Wrong: 620 / 1352
  ... never reached FINAL ANSWER: 13


,id,gold_answer,pred_numeric,pred_unit,gold_unit,reached_final
0,LD343,2.027*10^6,1.710000e+06,N/C,V/m,True
2,LD110,0.023,2.000000e+00,N,N,True
6,LD274,36.32,2.120000e+00,N,N,True
7,LD377,8.44 × 10^6,0.000000e+00,V/m,V/m,True
9,LD049,14.34,1.562500e+03,N,N,True
11,LD328,3.29*10^6,2.540000e+07,V/m,V/m,True
12,LD098,1.152 × 10^7,1.728000e+06,V/m,V/m,True
13,LD087,-2\sqrt{2} x q,2.000000e+00,C,-,True
15,LD385,3.50 × 10^6,3.840000e+06,V/m,V/m,True
16,LD083,9.6 × 10^2,2.250000e+04,V/m,V/m,True



  GOLDEN: app/physics_solution/versions/v03_routed_fewshot/output/results_golden.json
summary: {
  "n": 1352,
  "correct": 740,
  "accuracy": 0.5473372781065089,
  "mean_latency_s": 0.5366514790692979,
  "version": "v03_routed_fewshot",
  "model_id": "Qwen/Qwen3.5-4B",
  "dtype": "bf16",
  "effective_dtype": "bf16",
  "batch_size": 80,
  "assistant_prefix": "",
  "description": "Two-pass pipeline on the same model: Pass 1 classifies (domain, answer_type) from question text alone; Pass 2 solves with a domain-specific system prompt and answer-type-matched few-shot examples.",
  "classify_elapsed_s": 53.0311336517334,
  "n_examples": 3
}

Wrong: 612 / 1352
  ... never reached FINAL ANSWER: 14


,id,gold_answer,pred_numeric,pred_unit,gold_unit,reached_final
0,LD343,2.027 * 10^{6},1.710000e+06,N/C,V/m,True
2,LD110,0.023,2.000000e+00,N,N,True
6,LD274,36.32,2.120000e+00,N,N,True
7,LD377,8.44 * 10^{6},0.000000e+00,V/m,V/m,True
9,LD049,14.34,1.562500e+03,N,N,True
11,LD328,3.29 * 10^{6},2.540000e+07,V/m,V/m,True
12,LD098,1.152 * 10^{7},1.728000e+06,V/m,V/m,True
13,LD087,-2\sqrt{2} q,2.000000e+00,C,-,True
15,LD385,3.50 * 10^{6},3.840000e+06,V/m,V/m,True
16,LD083,9.6 * 10^{2},2.250000e+04,V/m,V/m,True


## 7. Routing accuracy check

Compare predicted domain (from Pass 1 classifier) vs the ID prefix ground truth.

In [ ]:
import re
PREFIX_RE = re.compile(r"^([A-Z]+)")

# Extract routing info from extra column
df['routed_domain'] = df['extra'].apply(lambda x: x.get('routed_domain', '') if isinstance(x, dict) else '')
df['routed_answer_type'] = df['extra'].apply(lambda x: x.get('routed_answer_type', '') if isinstance(x, dict) else '')
df['true_domain'] = df['id'].apply(lambda x: PREFIX_RE.match(str(x)).group(1) if PREFIX_RE.match(str(x)) else '')

df['domain_correct'] = df['routed_domain'] == df['true_domain']
print(f"Domain routing accuracy: {df['domain_correct'].sum()}/{len(df)} = {df['domain_correct'].mean():.3f}")

# Per-domain routing accuracy
print("\nPer-domain routing accuracy:")
for domain, sub in df.groupby('true_domain'):
    acc = sub['domain_correct'].mean()
    print(f"  {domain}: {sub['domain_correct'].sum()}/{len(sub)} = {acc:.3f}")

# Confusion: what domains were misclassified as what
misrouted = df[~df['domain_correct']]
if len(misrouted) > 0:
    print(f"\nMisrouted examples ({len(misrouted)} total):")
    print(pd.crosstab(misrouted['true_domain'], misrouted['routed_domain'], margins=True))

## 8. Deep error analysis

Open [`app/physics_solution/eda/notebooks/error_analysis.ipynb`](../../eda/notebooks/error_analysis.ipynb) — it categorises every wrong row into fail modes and writes a markdown report.